# Stage 3 — traffic forensic analysis

This notebook derives traffic-specific evidence from the canonical CDR table and compares known fraudulent and legitimate SIMs. It does not produce a final fraud decision.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PROCESSED=Path('../outputs/processed/all_naive_12pct_200sims')
OUTPUT=Path('../outputs/traffic/all_naive_12pct_200sims')
OUTPUT.mkdir(parents=True,exist_ok=True)
cdr=pd.read_csv(PROCESSED/'canonical_cdr.csv',dtype={'src_number':'string','dst_number':'string'},parse_dates=['timestamp'])
features=pd.read_csv('../outputs/statistical/all_naive_12pct_200sims/sim_statistical_features.csv',dtype={'sim_number':'string'})
print('canonical CDRs:',len(cdr),'SIM profiles:',len(features))

In [ ]:
voice={'MOC','MTC','I-MOC','I-MTC'}; outgoing={'MOC','I-MOC','MO-SMS'}; incoming={'MTC','I-MTC'}; international={'I-MOC','I-MTC'}

features['international_voice_activity_count']=features['international_activity_count']
features['activity_population_share']=features['activity_count']/features['activity_count'].sum()
features['outgoing_share']=features['outgoing_activity_count']/features['activity_count'].replace(0,np.nan)
features['incoming_share']=features['incoming_activity_count']/features['activity_count'].replace(0,np.nan)
features['contact_per_active_hour']=features['unique_contacts']/features['active_hours'].replace(0,np.nan)
features['duration_per_activity']=features['total_duration']/features['activity_count'].replace(0,np.nan)
features['short_call_count']=0

# Count short voice calls per source SIM as a traffic-specific indicator.
short_source=cdr[cdr.cdr_type.isin(voice)&(cdr.duration<=10)].groupby('src_number').size()
features['short_call_count']=features['sim_number'].map(short_source).fillna(0)
features['short_call_ratio']=features['short_call_count']/features['voice_activity_count'].replace(0,np.nan)

traffic_columns=['activity_count','outgoing_activity_count','incoming_activity_count','outgoing_share','incoming_share','total_duration','duration_per_activity','unique_contacts','contact_per_active_hour','international_activity_count','international_activity_ratio','short_call_count','short_call_ratio','activity_population_share']
features.to_csv(OUTPUT/'traffic_features.csv',index=False)
features.head()

In [ ]:
counts=features.known_fraud.value_counts().reindex([False,True],fill_value=0)
if (counts==0).any(): raise ValueError(f'Both classes are required; counts={counts.to_dict()}')

comparison=features.groupby('known_fraud',observed=False)[traffic_columns].median().reindex([False,True]).T
comparison.columns=['known_legitimate_median','known_fraudulent_median']
comparison['difference']=comparison.known_fraudulent_median-comparison.known_legitimate_median
comparison.to_csv(OUTPUT/'traffic_group_median_comparison.csv')

correlations=features[traffic_columns].corr(method='spearman')
correlations.to_csv(OUTPUT/'traffic_correlations_spearman.csv')
comparison

In [ ]:
# Robust outlier evidence: how far each SIM is from the legitimate population median/IQR.
legit=features.loc[~features.known_fraud,traffic_columns]
q1=legit.quantile(.25,numeric_only=True); q3=legit.quantile(.75,numeric_only=True); iqr=q3-q1
rows=[]
for _,row in features.iterrows():
    for col in traffic_columns:
        value=row[col]
        if pd.isna(value) or pd.isna(iqr[col]) or iqr[col]==0: continue
        direction='high' if value>q3[col] else 'low' if value<q1[col] else 'within_iqr'
        if direction!='within_iqr': rows.append({'sim_number':row.sim_number,'known_fraud':row.known_fraud,'feature':col,'value':value,'legitimate_q1':q1[col],'legitimate_q3':q3[col],'iqr_deviation':(value-q3[col])/iqr[col] if direction=='high' else (q1[col]-value)/iqr[col],'direction':direction})
evidence=pd.DataFrame(rows).sort_values('iqr_deviation',ascending=False)
evidence.to_csv(OUTPUT/'traffic_iqr_deviations.csv',index=False)
print('evidence rows:',len(evidence)); print(evidence.head(20))

Interpret traffic differences as investigative evidence only. A high IQR deviation is a reason to inspect a SIM, not proof of fraud.